# Block 2 - Project
Bogdan Filipchuk, Yunhee Choi, Gregorio A. Rolli, Artem Hrynov

## 1. Pore analysis


| Structure | Density (g/cm³) | ASA (Å²) | POAV (Å³) | Porosity |
| :--- | ---: | ---: | ---: | ---: |
| Mg-MOF-74 | 0.886275 | 222.828 | 837.198 | 0.61368 |
| NOTT-300 | 1.039260 | 453.452 | 1297.110 | 0.48999 |
| UTSA-20 | 0.882399 | 1307.510 | 3575.360 | 0.62907 |
| UTSA-80 | 0.678215 | 3114.620 | 9619.780 | 0.66798 |
| ZIF-8 | 0.909567 | 787.421 | 2488.250 | 0.49906 |

**Table 1.** Density, accessible surface area (ASA), probe-occupiable accessible volume (POAV) and porosity of the five structures.

The pore properties were computed with Zeo++ using a probe radius of 1.525 Å (100,000 samples). ASA and POAV are given per unit cell, and the porosity is the POAV volume fraction (POAV divided by the unit-cell volume).

## 2. Henry coefficients at 298 K

| CO₂ affinity rank (1 = highest) | Structure | CO₂ Henry coefficient (mol·kg⁻¹·Pa⁻¹) | N₂ Henry coefficient (mol·kg⁻¹·Pa⁻¹) |
| ---: | :--- | ---: | ---: |
| 1 | NOTT-300 | $1.85432 \times 10^{-4}$ | $1.05206 \times 10^{-5}$ |
| 2 | Mg-MOF-74 | $8.65972 \times 10^{-5}$ | $3.33212 \times 10^{-6}$ |
| 3 | UTSA-20 | $4.09127 \times 10^{-5}$ | $4.37742 \times 10^{-6}$ |
| 4 | UTSA-80 | $1.69838 \times 10^{-5}$ | $3.33977 \times 10^{-6}$ |
| 5 | ZIF-8 | $8.83454 \times 10^{-6}$ | 0 (pores inaccessible to N₂) |

**Table 2.** CO₂ and N₂ Henry coefficients at 298 K from Widom insertion, ranked by CO₂ affinity.

The Henry coefficient is the slope of the isotherm at zero loading, so it measures how strongly a single CO₂ molecule interacts with the framework: a larger value means a higher CO₂ affinity. Mg-MOF-74 has a slightly more negative Widom adsorption energy than NOTT-300 (−24.4 vs −23.7 kJ/mol), but a smaller Henry coefficient. We rank by the Henry coefficient because it includes both the interaction strength and how many favorable adsorption sites are available per kilogram of material, while the adsorption energy is only an average energy.

For ZIF-8, the pore check with the N₂ probe (radius 1.655 Å) found the pores blocked (accessible volume 0), so no N₂ adsorption was simulated and its N₂ Henry coefficient is 0. The ZIF-8 window (about 3.4 Å) is larger than CO₂ (kinetic diameter 3.3 Å) but smaller than N₂ (3.64 Å), so in this rigid-framework model only CO₂ can enter the pores.

## 3. Pure-component adsorption isotherms at 298 K 

In [ ]:
import ast
import csv
from pathlib import Path

import matplotlib.pyplot as plt

plt.rcdefaults()
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
})

DATA_DIR = Path("isotherms")
STRUCTURES = [
    ("MgMOF74", "Mg-MOF-74"),
    ("NOTT-300", "NOTT-300"),
    ("UTSA-20", "UTSA-20"),
    ("UTSA-80", "UTSA-80"),
    ("ZIF-8", "ZIF-8"),
]
GASES = [("CO2", r"$\mathrm{CO}_2$"), ("N2", r"$\mathrm{N}_2$")]


colors = [plt.colormaps["magma"](x) for x in (0.15, 0.32, 0.49, 0.65, 0.80)]
markers = ["o", "s", "^", "D", "v"]
figures = {}

for gas, gas_label in GASES:
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    figures[gas] = fig
    missing = []

    for (stem, label), color, marker in zip(STRUCTURES, colors, markers):
        filename = DATA_DIR / f"{stem}-{gas}.csv"
        with filename.open(newline="") as file:
            data = {row["Key"]: row["Value"] for row in csv.DictReader(file)}

        if "isotherm" not in data:
            missing.append(label)
            continue

        iso = ast.literal_eval(data["isotherm"])

        ax.errorbar(
            iso["pressure"],
            iso["loading_absolute_average"],
            yerr=iso["loading_absolute_dev"],
            label=label,
            color=color,
            marker=marker,
            linestyle="-",
            linewidth=1.2,
            markersize=4,
            markeredgewidth=0.7,
            capsize=2.5,
            elinewidth=0.8,
            capthick=0.8,
        )

    ax.set(
        title=f"{gas_label} adsorption at 298 K",
        xlabel="Pressure (bar)",
        ylabel=r"Absolute loading (mol kg$^{-1}$)",
        xlim=(0, 10.5),
        ylim=(0, None),
    )
    ax.set_xticks(range(0, 11, 2))
    ax.legend(frameon=False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="0.9", linewidth=0.6)
    ax.tick_params(direction="in", top=True, right=True)
    # Structures without an isotherm have pores that the molecule cannot enter (see task 2).
    if missing:
        ax.text(
            0.98, 0.03,
            f"{', '.join(missing)}: no {gas_label} uptake (pores inaccessible)",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8,
        )
    fig.tight_layout()

    # fig.savefig(f"{gas}_isotherms.pdf", bbox_inches="tight")
    # fig.savefig(f"{gas}_isotherms.png", dpi=300, bbox_inches="tight")

plt.show()


**Figure 1.** Pure-component CO₂ adsorption isotherms at 298 K from GCMC simulations (0.2–10.2 bar). Error bars show the statistical uncertainty of the simulations.

**Figure 2.** Pure-component N₂ adsorption isotherms at 298 K. ZIF-8 is not shown because N₂ cannot enter its pores.

## 4. CO₂ working capacity

Binary CO₂/N₂ equilibrium loadings were calculated using IAST at 298 K, with adsorption at 1.0 bar and desorption at 0.2 bar. The gas composition is assumed to remain 15% CO₂ / 85% N₂ at both conditions. Thus, the CO₂/N₂ partial pressures are 0.15/0.85 bar for adsorption and 0.03/0.17 bar for desorption.

In [ ]:
import ast
import csv
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from scipy.optimize import brentq, least_squares


@dataclass
class QuadraticIsotherm:
    m: float
    a: float
    b: float
    max_pressure: float
    rmse: float = 0.0

    def loading(self, pressure):
        p = np.asarray(pressure)
        return self.m * (self.a * p + 2 * self.b * p**2) / (
            1 + self.a * p + self.b * p**2
        )

    def spreading_pressure(self, pressure):
        return self.m * np.log1p(self.a * pressure + self.b * pressure**2)


def fit_isotherm(path):
    with path.open(newline="") as file:
        data = {row["Key"]: row["Value"] for row in csv.DictReader(file)}
    if "isotherm" not in data:
        return None
    iso = ast.literal_eval(data["isotherm"])
    p = np.array(iso["pressure"])
    q = np.array(iso["loading_absolute_average"])
    dev = np.array(iso["loading_absolute_dev"])
    kh = float(data["henry_coefficient_average"]) * 1e5  # Pa⁻¹ -> bar⁻¹

    def model(log_parameters):
        m, b = np.exp(log_parameters)
        return QuadraticIsotherm(m, kh / m, b, p.max())
    
    fits = [
        least_squares(
            lambda t: (model(t).loading(p) - q) / dev,
            np.log([q.max() / 2, initial_b]),
            bounds=(-25, 25), max_nfev=3000,
        )
        for initial_b in (0.001, 0.1, 1, 10)
    ]
    successful = [fit for fit in fits if fit.success]
    best = min(successful, key=lambda fit: np.sum(fit.fun**2))
    fitted = model(best.x)
    fitted.rmse = float(np.sqrt(np.mean((fitted.loading(p) - q)**2)))
    return fitted


def binary_iast(partial_pressures, models):
    partial = np.asarray(partial_pressures, dtype=float)
    low = partial[0] / models[0].max_pressure
    high = 1 - partial[1] / models[1].max_pressure

    def residual(x):
        return (models[0].spreading_pressure(partial[0] / x)
                - models[1].spreading_pressure(partial[1] / (1 - x)))

    x_co2 = brentq(residual, low, high, xtol=1e-13)
    x = np.array([x_co2, 1 - x_co2])
    reference_pressures = partial / x
    pure_loadings = np.array([
        model.loading(p) for model, p in zip(models, reference_pressures)
    ])
    loadings = x / np.sum(x / pure_loadings)
    return loadings, reference_pressures


def calculate(data_dir=Path("isotherms")):
    results = []
    for stem in ("MgMOF74", "NOTT-300", "UTSA-20", "UTSA-80", "ZIF-8"):
        models = [fit_isotherm(data_dir / f"{stem}-{gas}.csv") for gas in ("CO2", "N2")]
        result = {"structure": stem.replace("MgMOF74", "Mg-MOF-74"), "models": models}
        if all(model is not None for model in models):
            result["ads"], result["p0_ads"] = binary_iast([0.15, 0.85], models)
            result["des"], result["p0_des"] = binary_iast([0.03, 0.17], models)
            result["wc"] = result["ads"][0] - result["des"][0]
        elif models[0] is not None:
            # N₂ cannot enter the pores (task 2), so N₂ loading is zero and
            # IAST reduces to the pure CO₂ isotherm at the CO₂ partial pressure.
            result["ads"] = np.array([models[0].loading(0.15), 0.0])
            result["des"] = np.array([models[0].loading(0.03), 0.0])
            result["wc"] = result["ads"][0] - result["des"][0]
        results.append(result)
    return results


def markdown_table(results):
    lines = [
        "| Structure | CO₂ loading at 1.0 bar (mmol/g) | CO₂ loading at 0.2 bar (mmol/g) | CO₂ working capacity (mmol/g) |",
        "| :--- | ---: | ---: | ---: |",
    ]
    for row in results:
        if "wc" in row:
            lines.append(f"| {row['structure']} | {row['ads'][0]:.3f} | "
                         f"{row['des'][0]:.3f} | {row['wc']:.3f} |")
        else:
            lines.append(f"| {row['structure']} | N/A | N/A | N/A |")
    return "\n".join(lines)

In [ ]:
from IPython.display import Markdown, display

working_capacity_results = calculate()
display(Markdown(markdown_table(working_capacity_results)))

**Table 3.** IAST CO₂ loadings at adsorption (1.0 bar) and desorption (0.2 bar) conditions for 15% CO₂ / 85% N₂ at 298 K, and the resulting CO₂ working capacity.

**Procedure**

1. We used a quadratic isotherm model to describe how loading changes with pressure. The Henry coefficient fixes the slope near zero pressure, while the remaining parameters are fitted to the simulated data. 
2. We used IAST to combine the two pure-gas curves and account for CO₂ and N₂ competing for adsorption. We calculated the CO₂ loading at total pressures of 1.0 bar and 0.2 bar, assuming 15% CO₂ and 85% N₂ at both pressures. For ZIF-8, N₂ cannot enter the pores (task 2), so there is no competition: the N₂ loading is zero and the CO₂ loading is taken directly from the pure CO₂ isotherm at 0.15 bar and 0.03 bar.
3. The working capacity is the amount adsorbed at 1.0 bar minus the amount remaining adsorbed at 0.2 bar:

$$WC = q_{\mathrm{CO_2}}(1.0\,\mathrm{bar}) - q_{\mathrm{CO_2}}(0.2\,\mathrm{bar}).$$

The result is reported in mmol/g, which has the same numerical value as the mol/kg units in the input files.

## 5. CO₂/N₂ selectivity

We use the binary CO₂ and N₂ loadings calculated by IAST in task 4 at 1.0 bar and 298 K. For a gas mixture containing 15% CO₂ and 85% N₂, selectivity is calculated as

$$S_{\mathrm{CO_2/N_2}} = \frac{q_{\mathrm{CO_2}}}{q_{\mathrm{N_2}}}\frac{y_{\mathrm{N_2}}}{y_{\mathrm{CO_2}}} = \frac{q_{\mathrm{CO_2}}}{q_{\mathrm{N_2}}}\frac{0.85}{0.15}.$$

Here $q_i$ is the mixture loading at adsorption conditions and $y_i$ is the gas-phase mole fraction. For ZIF-8, $q_{\mathrm{N_2}} = 0$, so its selectivity is infinite.

In [ ]:
from IPython.display import Markdown, display


y_co2, y_n2 = 0.15, 0.85
selectivity_results = []
lines = [
    "| Structure | CO₂ loading at 1.0 bar (mmol/g) | N₂ loading at 1.0 bar (mmol/g) | CO₂/N₂ selectivity (dimensionless) |",
    "| :--- | ---: | ---: | ---: |",
]

for row in working_capacity_results:
    if "ads" not in row:
        selectivity_results.append({"structure": row["structure"], "selectivity": None})
        lines.append(f"| {row['structure']} | N/A | N/A | N/A |")
        continue

    q_co2, q_n2 = row["ads"]
    # N₂ is excluded from the pores (ZIF-8), so the selectivity is infinite.
    selectivity = (q_co2 / q_n2) * (y_n2 / y_co2) if q_n2 > 0 else float("inf")
    selectivity_results.append({
        "structure": row["structure"],
        "q_co2": q_co2,
        "q_n2": q_n2,
        "selectivity": selectivity,
    })
    lines.append(
        f"| {row['structure']} | {q_co2:.3f} | {q_n2:.3f} | {selectivity:.3f} |"
    )

display(Markdown("\n".join(lines).replace("| inf |", "| ∞ |")))

**Table 4.** IAST CO₂ and N₂ loadings at adsorption conditions (1.0 bar, 298 K) and the resulting CO₂/N₂ selectivity.

## 6. Ranking using working capacity and selectivity

We plot CO₂ working capacity against CO₂/N₂ selectivity using the results from tasks 4 and 5. Higher values on both axes are preferred, so the most favorable region is the upper right. The calculation uses 298 K, adsorption at 1.0 bar, and desorption at 0.2 bar, with 15% CO₂ / 85% N₂ at both pressures.

In [ ]:
import math

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.rcdefaults()
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
})

# Run tasks 4 and 5 first; retain full precision for plotting and ranking.
structure_order = ["Mg-MOF-74", "NOTT-300", "UTSA-20", "UTSA-80", "ZIF-8"]
colors = [plt.colormaps["magma"](x) for x in (0.15, 0.32, 0.49, 0.65, 0.80)]
markers = ["o", "s", "^", "D", "v"]
wc_by_structure = {row["structure"]: row.get("wc") for row in working_capacity_results}
s_by_structure = {row["structure"]: row["selectivity"] for row in selectivity_results}
# Infinite selectivity (ZIF-8) is drawn at the right edge of the axes.
x_max = max(s for s in s_by_structure.values() if s is not None and not math.isinf(s)) * 1.18

screening_results = []
fig_screening, ax = plt.subplots(figsize=(6, 4), dpi=300)
for name, color, marker in zip(structure_order, colors, markers):
    wc = wc_by_structure[name]
    selectivity = s_by_structure[name]
    if wc is None or selectivity is None:
        continue
    screening_results.append({"structure": name, "wc": wc, "selectivity": selectivity})
    x = x_max if math.isinf(selectivity) else selectivity
    ax.plot(
        x, wc,
        color=color, marker=marker, linestyle="none",
        markersize=7, markeredgewidth=0.7, clip_on=False,
    )
   
    right_side = selectivity > 20
    ax.annotate(
        f"{name} (S → ∞)" if math.isinf(selectivity) else name, (x, wc),
        xytext=(-10 if right_side else 10, 8),
        textcoords="offset points",
        ha="right" if right_side else "left", va="bottom",
        fontsize=9, color=color,
    )

ax.set(
    title=r"CO$_2$ capture screening at 298 K",
    xlabel=r"CO$_2$/N$_2$ selectivity",
    ylabel=r"CO$_2$ working capacity (mmol g$^{-1}$)",
    xlim=(0, x_max),
    ylim=(0, max(row["wc"] for row in screening_results) * 1.18),
)
ax.set_axisbelow(True)
ax.tick_params(direction="in", top=True, right=True)
fig_screening.tight_layout()

# fig_screening.savefig("working_capacity_vs_selectivity.pdf", bbox_inches="tight")
# fig_screening.savefig("working_capacity_vs_selectivity.png", dpi=300, bbox_inches="tight")

plt.show()

ranked_results = sorted(screening_results, key=lambda row: row["wc"], reverse=True)

lines = [
    "| Rank | Structure | CO₂ working capacity (mmol/g) | CO₂/N₂ selectivity |",
    "| ---: | :--- | ---: | ---: |",
]
for rank, row in enumerate(ranked_results, start=1):
    lines.append(f"| {rank} | {row['structure']} | {row['wc']:.3f} | {row['selectivity']:.3f} |")
for name in structure_order:
    if wc_by_structure[name] is None or s_by_structure[name] is None:
        lines.append(f"| N/A | {name} | N/A | N/A |")
display(Markdown("\n".join(lines).replace("| inf |", "| ∞ |")))


**Figure 3.** CO₂ working capacity versus CO₂/N₂ selectivity at 298 K. ZIF-8 has infinite selectivity and is drawn at the right edge of the axes.

**Table 5.** Ranking of the structures based on CO₂ working capacity and CO₂/N₂ selectivity.

For these model estimates, the ranking is **NOTT-300 > Mg-MOF-74 > UTSA-20 > UTSA-80 > ZIF-8**. From NOTT-300 to UTSA-80, each successive structure has lower working capacity and lower selectivity. NOTT-300 and Mg-MOF-74 have practically equal selectivity (27.339 and 27.336); their separation in this ranking is mainly due to working capacity. With the same selectivity, NOTT-300 recovers about 2.6 times more CO₂ per gram of adsorbent in each cycle (2.642 vs 1.012 mmol/g), so less adsorbent is needed for the same amount of captured CO₂. ZIF-8 is the exception: N₂ cannot enter its pores, so its selectivity is infinite (shown at the right edge of the plot), but its working capacity (0.109 mmol/g) is the lowest. We rank it last because a high selectivity is of little use when very little CO₂ is recovered per cycle. 

## 7. Discussion

Our screening suggests that NOTT-300 is the most promising of the five materials we compared, mainly because of its higher working capacity. However, this is not enough to conclude that it would perform best in a real PSA/VSA process.

We screened the materials at the molecular and equilibrium level, using pure-gas simulations and IAST to estimate mixture adsorption. We did not simulate a full adsorption cycle. The calculation assumes constant temperature and the same gas composition during adsorption and desorption. It also uses fitted curves below the lowest simulated pressure. Adsorption rates, heat effects, pressure losses, and water in real flue gas are not included. The results also depend on the force field: the generic UFF force field used here does not describe the strong binding of CO₂ to the open metal sites of Mg-MOF-74 well, so its CO₂ affinity is probably underestimated. Experimentally, Mg-MOF-74 is one of the best MOFs for CO₂ adsorption at low pressure, so the order of NOTT-300 and Mg-MOF-74 could change with a more accurate model. The infinite selectivity of ZIF-8 comes from treating the framework as rigid: in reality its linkers can rotate and widen the windows, so N₂ is also adsorbed.

A more complete comparison could use these process indicators:

- CO₂ purity: the mole fraction of CO₂ in the collected product.
- CO₂ recovery: the fraction of CO₂ entering with the feed that is collected in the product.
- Productivity: the amount of CO₂ collected per unit mass of adsorbent per unit time, including all cycle steps.
- Specific energy consumption: the vacuum and compression energy used per unit mass of CO₂ captured.

We would compare energy consumption and productivity while requiring a minimum purity and recovery, for example the 95% purity and 90% recovery targets used by Burns et al. [1]. This needs a dynamic process model that follows the bed through repeated adsorption and regeneration steps, including heat and mass transfer. Burns et al. found that reliable energy and productivity estimates require full process simulations [1].


## References

1. T. D. Burns et al., "Prediction of MOF Performance in Vacuum Swing Adsorption Systems for Postcombustion CO₂ Capture Based on Integrated Molecular Simulations, Process Optimizations, and Machine Learning Models," *Environ. Sci. Technol.*, vol. 54, no. 7, pp. 4536–4544, 2020. https://doi.org/10.1021/acs.est.9b07407